# Python Mastery — Module 8
# Deep Learning Foundations: Beginner to Advanced

A practical introduction to neural networks, tensors, forward propagation, loss functions, gradient descent, backpropagation, optimizers, regularization, and reliable training workflows.

## Learning outcomes
By the end, you should be able to:
- Explain neurons, layers, weights, biases, and activation functions.
- Understand tensors and common tensor shapes.
- Trace forward propagation from inputs to predictions.
- Choose loss functions for regression and classification.
- Explain gradients, gradient descent, and backpropagation.
- Compare SGD, momentum, RMSProp, and Adam conceptually.
- Diagnose overfitting, underfitting, unstable training, and vanishing/exploding gradients.
- Use validation data, early stopping, regularization, and learning curves.
- Train and evaluate a small neural network with scikit-learn and, optionally, PyTorch.
- Understand when a neural network is or is not the right choice.

**Examples use synthetic data.** They teach methods and do not establish real-world clinical or business effectiveness.

## 1. Where deep learning fits

Machine learning learns patterns from data. Deep learning uses neural networks with multiple layers of learnable transformations.

A neural network can be useful when relationships are complex or data are unstructured (images, audio, text). For many tabular business datasets, gradient-boosted trees and simpler models remain highly competitive, easier to tune, and easier to explain.

A good workflow still needs:
1. a clearly defined target and prediction time;
2. leakage-safe splits;
3. a baseline;
4. a suitable loss and metric;
5. validation-based model selection;
6. a final evaluation on untouched test data.

## 2. A neuron and a dense layer

A simple neuron calculates a weighted sum of its inputs, adds a bias, and applies an activation function.

\[
z = w_1x_1 + w_2x_2 + \cdots + w_dx_d + b
\]

\[
a = f(z)
\]

- **Weights:** learned strengths of connections.
- **Bias:** learned offset.
- **Activation:** introduces a nonlinearity.
- **Layer:** a collection of neurons operating on inputs.
- **Dense / fully connected layer:** each output unit receives input from every unit in the previous layer.

Without nonlinear activations, stacking dense layers still produces an overall linear transformation, so depth alone would not create a rich nonlinear model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# One neuron with two inputs.
x = np.array([2.0, -1.0])
w = np.array([0.7, 0.4])
b = 0.2
z = np.dot(x, w) + b
relu = max(0.0, z)

print("Weighted sum plus bias:", z)
print("ReLU activation:", relu)

## 3. Common activation functions

- **ReLU:** `max(0, z)`. Simple and widely used in hidden layers.
- **Sigmoid:** maps values to (0, 1), often used as the output activation for binary probabilities in simple explanations.
- **Tanh:** maps values to (−1, 1).
- **Softmax:** converts a vector of class scores into values that sum to 1, commonly for mutually exclusive multi-class classification.

In many modern frameworks, the output layer returns logits and the loss function handles the appropriate sigmoid/softmax calculation internally for numerical stability.

In [ ]:
z = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)

plt.figure(figsize=(8, 4.5))
plt.plot(z, sigmoid, label="Sigmoid")
plt.plot(z, tanh, label="Tanh")
plt.plot(z, relu, label="ReLU")
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.xlabel("z")
plt.ylabel("Activation")
plt.title("Common activation functions")
plt.legend()
plt.show()

## 4. Tensors and shapes

A tensor is a general container for numerical values:
- scalar: zero-dimensional tensor;
- vector: one-dimensional tensor;
- matrix: two-dimensional tensor;
- higher-dimensional tensor: three or more dimensions.

In a typical tabular neural network:
- `X` has shape `(number_of_samples, number_of_features)`;
- a binary target may have shape `(number_of_samples,)` or `(number_of_samples, 1)`;
- a batch is a subset of samples processed together.

Shape errors are among the most common deep-learning bugs. Check the batch dimension, feature dimension, data type, and device (CPU/GPU) when using a framework.

In [ ]:
scalar = np.array(3.14)
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
tensor3d = np.zeros((8, 4, 3))

for name, value in [
    ("scalar", scalar), ("vector", vector),
    ("matrix", matrix), ("3D tensor", tensor3d)
]:
    print(f"{name:10s} shape={value.shape}, ndim={value.ndim}")

## 5. Forward propagation

Forward propagation passes inputs through the layers to produce predictions.

For a two-layer network:
1. calculate the hidden layer's weighted sum and activation;
2. pass hidden activations to the output layer;
3. convert output scores into predictions or probabilities as appropriate.

The parameters are initially set (often randomly) and then updated during training to reduce the loss. A prediction from an untrained network is not expected to be useful.

In [ ]:
# A tiny forward pass using NumPy, for shape intuition.
rng = np.random.default_rng(42)
X_demo = rng.normal(size=(5, 3))  # 5 samples, 3 features
W1 = rng.normal(scale=0.3, size=(3, 4))  # 3 input features -> 4 hidden units
b1 = np.zeros((1, 4))
W2 = rng.normal(scale=0.3, size=(4, 1))  # 4 hidden units -> 1 output
b2 = np.zeros((1, 1))

hidden = np.maximum(0, X_demo @ W1 + b1)
logits = hidden @ W2 + b2
print("Input shape:", X_demo.shape)
print("Hidden shape:", hidden.shape)
print("Output logits shape:", logits.shape)

## 6. Loss functions

A loss quantifies how poorly a model's predictions match the targets. Training attempts to minimize the loss.

- **Mean squared error (MSE):** common for regression; penalizes large errors quadratically.
- **Mean absolute error (MAE):** regression loss less sensitive to large errors than MSE.
- **Binary cross-entropy / log loss:** common for binary classification.
- **Categorical cross-entropy:** common for mutually exclusive multi-class classification.
- **Sparse categorical cross-entropy:** similar multi-class objective when targets are integer class IDs.

Choose the loss to match the output representation and task. For classification, do not confuse the training loss with the business evaluation metric; the model may be trained with log loss but evaluated with recall, average precision, or another metric.

In [ ]:
y_true = np.array([0, 1, 1, 0])
y_pred_reg = np.array([0.2, 0.8, 0.6, 0.4])
y_prob = np.clip(np.array([0.1, 0.75, 0.65, 0.3]), 1e-7, 1 - 1e-7)

mse = np.mean((y_true - y_pred_reg) ** 2)
mae = np.mean(np.abs(y_true - y_pred_reg))
binary_cross_entropy = -np.mean(
    y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)
)
print("MSE:", round(mse, 4))
print("MAE:", round(mae, 4))
print("Binary cross-entropy:", round(binary_cross_entropy, 4))

## 7. Gradients and gradient descent

A gradient describes how a function changes with its parameters. Gradient descent updates parameters in the direction that decreases the loss.

Conceptually:

\[
\theta_{new}=\theta_{old}-\eta\nabla_\theta L
\]

- `θ`: model parameters (weights and biases);
- `L`: loss;
- `∇L`: gradient of loss with respect to parameters;
- `η`: learning rate.

If the learning rate is too high, training may oscillate or diverge. If too low, training can be very slow or appear stuck. Learning-rate schedules can reduce the learning rate as training progresses.

## 8. Backpropagation: how gradients are calculated

Backpropagation applies the chain rule to compute how each parameter contributed to the loss. It efficiently calculates gradients layer by layer, from the output back toward earlier layers.

A training step commonly follows:
1. forward pass;
2. calculate loss;
3. calculate gradients by backpropagation;
4. optimizer updates parameters;
5. repeat for batches and epochs.

**Backpropagation calculates gradients; the optimizer uses those gradients to update parameters.** These are related but different operations.

In [ ]:
# A numerical demonstration of a gradient for a single parameter.
# Suppose L(w) = (w*x - y)^2 for one data point.
x_value, y_value, w_value = 2.0, 5.0, 1.0
prediction = w_value * x_value
loss = (prediction - y_value) ** 2
gradient = 2 * (prediction - y_value) * x_value

learning_rate = 0.05
w_new = w_value - learning_rate * gradient
new_loss = (w_new * x_value - y_value) ** 2

print("Old weight:", w_value, "old loss:", loss)
print("Gradient:", gradient)
print("Updated weight:", w_new, "new loss:", new_loss)

## 9. Batch, stochastic, and mini-batch gradient descent

- **Batch gradient descent:** calculates the gradient using the full training dataset for each update.
- **Stochastic gradient descent (SGD):** updates using one observation at a time.
- **Mini-batch gradient descent:** uses a small batch of observations; common in deep learning because it balances computation efficiency and gradient noise.

An **epoch** is one pass through the training dataset. The number of batches per epoch depends on batch size and dataset size.

## 10. Optimizers: SGD, momentum, RMSProp, and Adam

- **SGD:** simple gradient-based updates; performance depends on learning rate and schedule.
- **Momentum:** accumulates a running direction from recent gradients to reduce zig-zagging and accelerate consistent movement.
- **RMSProp:** adapts step sizes using a moving average of squared gradients.
- **Adam:** combines momentum-like first-moment tracking with adaptive scaling based on squared gradients.

Adam is a useful default for many neural-network tasks, but it is not always best. Optimizer choice, learning rate, batch size, and architecture interact; compare using validation performance.

## 11. Why initialization matters

If every hidden neuron starts with identical weights, the neurons can learn identical features. Random initialization breaks this symmetry. Common initializations such as Xavier/Glorot and He initialization account for layer dimensions and activation functions to help keep activation and gradient scales reasonable.

Modern frameworks usually choose sensible defaults, but initialization still matters in very deep networks or custom architectures.

## 12. Vanishing and exploding gradients

During backpropagation, gradients are multiplied through layers. In very deep networks this can cause:
- **Vanishing gradients:** gradients become extremely small, so early layers learn very slowly.
- **Exploding gradients:** gradients become extremely large, causing unstable updates or numerical problems.

Potential mitigations include suitable initialization, ReLU-like activations, normalization, residual connections, careful learning rates, and gradient clipping. No single technique solves every architecture or data issue.

## 13. Regularization and generalization

A neural network can memorize training examples rather than learn patterns that generalize.

Common controls:
- **Weight decay / L2 regularization:** discourages excessively large weights.
- **Dropout:** randomly disables some activations during training to discourage co-adaptation.
- **Early stopping:** stops training when validation performance stops improving.
- **Data augmentation:** creates valid variations of training examples, especially in image/audio tasks.
- **Simpler architecture:** fewer parameters can reduce overfitting.
- **More representative data:** often helps more than endlessly tuning the architecture.

Dropout is typically disabled during evaluation. Regularization settings should be selected using validation data, not the test set.

## 14. Training, validation, and test sets for deep learning

- Training data update the model's weights.
- Validation data guide architecture, hyperparameters, early stopping, and threshold decisions.
- Test data estimate final performance after choices are complete.

Keep the split aligned with deployment: stratified for imbalanced classification, group-based for repeated entities, and time-based for future prediction. For small datasets, use cross-validation where appropriate.

Fit scaling and other learned preprocessing on training data only.

## 15. First practical model: an MLP with scikit-learn

A **multilayer perceptron (MLP)** is a feed-forward neural network. This first end-to-end example uses scikit-learn's `MLPClassifier` to keep the code accessible.

Notes:
- scale input features;
- start with a baseline;
- monitor convergence warnings and validation performance;
- MLPs can be sensitive to scaling, learning rate, architecture, and sample size;
- for larger or more customized deep-learning workflows, frameworks such as PyTorch or TensorFlow are commonly used.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss, classification_report
)

RANDOM_STATE = 42
X, y = make_classification(
    n_samples=3000, n_features=20, n_informative=10, n_redundant=4,
    weights=[0.85, 0.15], class_sep=1.0, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
print("Baseline accuracy:", round(baseline.score(X_test, y_test), 3))

In [ ]:
mlp = Pipeline([
    ("scale", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size=64,
        learning_rate_init=0.001,
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=12,
        random_state=RANDOM_STATE
    )),
])
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
mlp_proba = mlp.predict_proba(X_test)[:, 1]

mlp_results = {
    "accuracy": accuracy_score(y_test, mlp_pred),
    "precision": precision_score(y_test, mlp_pred, zero_division=0),
    "recall": recall_score(y_test, mlp_pred, zero_division=0),
    "F1": f1_score(y_test, mlp_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, mlp_proba),
    "average_precision": average_precision_score(y_test, mlp_proba),
    "log_loss": log_loss(y_test, mlp.predict_proba(X_test)),
}
pd.Series(mlp_results).round(3)

### Inspect the training curve

The MLP's `loss_curve_` shows training loss across iterations. When early stopping is enabled, `validation_scores_` records validation accuracy at each iteration. These curves can help diagnose convergence but do not replace the final untouched test evaluation.

In [ ]:
mlp_model = mlp.named_steps["model"]
plt.figure(figsize=(7, 4))
plt.plot(mlp_model.loss_curve_, label="Training loss")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("MLP training loss curve")
plt.legend()
plt.show()

if hasattr(mlp_model, "validation_scores_") and mlp_model.validation_scores_:
    plt.figure(figsize=(7, 4))
    plt.plot(mlp_model.validation_scores_, label="Internal validation accuracy")
    plt.xlabel("Iteration")
    plt.ylabel("Validation accuracy")
    plt.title("MLP early-stopping validation curve")
    plt.legend()
    plt.show()

## 16. Practical model tuning

Tune a few important parameters at a time:
- `hidden_layer_sizes`: network depth and width;
- `alpha`: L2 regularization strength;
- `learning_rate_init`: initial learning rate;
- `batch_size`: number of examples per update;
- activation function;
- early-stopping settings.

Use cross-validation or a dedicated validation set. Neural networks can be computationally expensive, so start with a small search. Compare against a simpler baseline and do not assume a deeper network is automatically better.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

tuning_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", MLPClassifier(
        max_iter=120, early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=10, random_state=RANDOM_STATE
    )),
])
param_grid = {
    "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "model__alpha": [0.0001, 0.01],
    "model__learning_rate_init": [0.001, 0.01],
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# A small illustrative grid; expand only if compute time permits.
search = GridSearchCV(
    tuning_pipeline, param_grid, scoring="average_precision",
    cv=cv, n_jobs=-1, refit=True
)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best CV average precision:", round(search.best_score_, 3))
print("Final held-out test AP:", round(average_precision_score(y_test, search.best_estimator_.predict_proba(X_test)[:, 1]), 3))

## 17. Optional PyTorch introduction

PyTorch is useful when you need more control over model architecture, custom losses, training loops, GPU execution, and research-style workflows.

This section is optional because PyTorch may not be installed in every environment. If import fails, install it using the official installation instructions for your operating system and CPU/GPU setup. The code below uses CPU and a small synthetic regression task to avoid requiring a GPU.

Concepts:
- `Tensor`: numerical data container;
- `nn.Module`: model definition;
- `DataLoader`: batches and shuffles examples;
- loss function: measures prediction error;
- optimizer: updates parameters;
- `zero_grad()`, `backward()`, `step()`: common training-loop operations.

In [ ]:
# Optional section: this cell checks whether PyTorch is installed.
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    print("PyTorch version:", torch.__version__)
except ImportError:
    print("PyTorch is not installed. You can skip the PyTorch section or install it separately.")

### Optional PyTorch training loop

If PyTorch imported successfully above, run the following cell. It trains a small MLP on synthetic regression data. The loop shows the essential sequence of a training step.

In [ ]:
if "torch" in globals():
    from sklearn.metrics import mean_absolute_error
    from sklearn.datasets import make_regression
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler

    X_pt, y_pt = make_regression(n_samples=1200, n_features=8, noise=15, random_state=RANDOM_STATE)
    X_pt_train, X_pt_test, y_pt_train, y_pt_test = train_test_split(
        X_pt, y_pt, test_size=0.2, random_state=RANDOM_STATE
    )
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_pt_train = x_scaler.fit_transform(X_pt_train).astype("float32")
    X_pt_test = x_scaler.transform(X_pt_test).astype("float32")
    y_pt_train = y_scaler.fit_transform(y_pt_train.reshape(-1, 1)).astype("float32")
    y_pt_test_scaled = y_scaler.transform(y_pt_test.reshape(-1, 1)).astype("float32")

    train_ds = TensorDataset(torch.tensor(X_pt_train), torch.tensor(y_pt_train))
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    class SmallMLP(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Linear(16, 1),
            )
        def forward(self, x):
            return self.network(x)

    model = SmallMLP(X_pt_train.shape[1])
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(40):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred_batch = model(xb)
            loss = criterion(pred_batch, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:02d}, train MSE (scaled target): {running_loss / len(train_ds):.4f}")

    model.eval()
    with torch.no_grad():
        test_pred_scaled = model(torch.tensor(X_pt_test)).numpy()
    test_pred_original = y_scaler.inverse_transform(test_pred_scaled).ravel()
    print("PyTorch test MAE (original target units):", round(mean_absolute_error(y_pt_test, test_pred_original), 2))
else:
    print("Skipped because PyTorch is not installed.")

## 18. Common training problems and how to investigate them

| Symptom | Possible causes | First checks |
|---|---|---|
| Loss barely changes | Learning rate too low, bad features, gradient issue, implementation bug | Check shapes, labels, loss, learning rate |
| Loss explodes or becomes NaN | Learning rate too high, bad inputs, numerical instability | Inspect data ranges, lower learning rate, consider gradient clipping |
| Training improves but validation worsens | Overfitting | Early stopping, regularization, smaller model, more data |
| Both training and validation are poor | Underfitting, wrong target/features, optimization issue | Improve features, model capacity, training setup |
| Performance varies widely by seed | Small dataset, unstable training, weak signal | Repeat runs, use robust validation, simplify |
| Excellent scores that seem unrealistic | Leakage, duplicates, split mismatch | Audit target timing and the full data pipeline |

These are hypotheses to investigate, not automatic diagnoses.

## 19. Exercises

Try these before reading the suggested answers.

1. What are weights and biases in a neural network?
2. Why do hidden layers need nonlinear activation functions?
3. Explain the difference between forward propagation and backpropagation.
4. What is the role of a loss function?
5. What happens when the learning rate is too high?
6. Distinguish an epoch from a batch.
7. How do dropout and early stopping reduce overfitting?
8. Why should features often be scaled before training an MLP?
9. Why is validation data used separately from the test set?
10. Why might a tree-based model outperform a neural network on tabular data?

### Suggested answers

1. Weights scale inputs; biases add learned offsets before activation.
2. Without nonlinear activations, a stack of dense layers collapses to a single linear transformation.
3. Forward propagation computes predictions; backpropagation computes gradients of loss with respect to parameters.
4. It quantifies mismatch between predictions and targets and provides the objective optimized during training.
5. Updates can overshoot good parameter values, causing oscillation or divergence.
6. An epoch is one pass over the training set; a batch is the subset processed in one update step.
7. Dropout randomly masks activations during training; early stopping halts when validation performance stops improving.
8. Different feature scales can make optimization poorly conditioned and cause some features to dominate.
9. Validation data support model selection; the test set should remain untouched until final evaluation.
10. Tabular datasets may have limited sample sizes, nonlinear threshold patterns, and mixed feature relationships that tree ensembles handle efficiently.

## 20. Capstone — train and evaluate a neural network

### Scenario
Build a binary classifier for a rare event using synthetic tabular data.

### Deliverables
1. Define the target, prediction moment, and unit of analysis.
2. Split the data with stratification.
3. Establish a dummy baseline.
4. Scale features inside a pipeline.
5. Train a small MLP with early stopping.
6. Report precision, recall, F1, ROC-AUC, average precision, and log loss.
7. Inspect training/validation curves.
8. Compare results against a simpler model from Module 6.
9. Describe overfitting risks, threshold trade-offs, and limitations.
10. Write a brief model card and monitoring plan.

Do not choose the final threshold or architecture by repeatedly inspecting the test set.

## 21. Readiness checklist

- [ ] Explain a neuron, weights, bias, layers, and activation.
- [ ] Read tensor shapes and diagnose simple dimension mismatches.
- [ ] Explain forward propagation and loss functions.
- [ ] Explain gradients, gradient descent, and backpropagation.
- [ ] Distinguish batch, epoch, and mini-batch training.
- [ ] Explain SGD, momentum, RMSProp, and Adam.
- [ ] Explain initialization and vanishing/exploding gradients.
- [ ] Apply regularization, early stopping, and validation correctly.
- [ ] Train and evaluate an MLP with an appropriate baseline.
- [ ] Diagnose underfitting, overfitting, and unstable training.
- [ ] Explain when deep learning may not be the best choice.

### Next module
**Module 9 — Deep Learning for Computer Vision:** CNNs, convolution, filters, padding, stride, pooling, image preprocessing, transfer learning, augmentation, and practical image classification.